In [1]:
# importa as bilbiotecas

import requests as rq
import urllib3
import pandas as pd

In [2]:
#faz o request
# Evita warning de certificado SSL (para testes locais)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = 'https://api-comexstat.mdic.gov.br/general'

headers = {
    'Accept': 'application/json',
    'Content-Type': 'application/json'
}

body = {
   "flow": "export",
   "monthDetail": True,
   "period":{
       "from": "2025-01",
       "to": "2025-12"
   },
   "filters":[
       {
           "filter":"heading",
           "values":[6401,6402,6403,6404,6405]
       }
   ],
   "details":[
       "country",
       "state",
       "ncm",
       "heading"
   ],
   "metrics":[
       "metricFOB",
       "metricKG",
       "metricStatistic"
   ]

}

response = rq.post(url, headers=headers, json=body, verify=False)

print("Status Code:", response.status_code)
print(response.text)


Status Code: 200
{"data":{"list":[{"coNcm":"64039990","year":"2025","monthNumber":"02","country":"Estados Unidos","state":"Rio Grande do Sul","ncm":"Outros cal\u00e7ados sola exterior borracha\/pl\u00e1stico, de couro\/natural","headingCode":"6403","heading":"Cal\u00e7ado com sola exterior de borracha, pl\u00e1stico, couro natural ou reconstitu\u00eddo e parte superior de couro natural","metricFOB":"6338562","metricKG":"133474","metricStatistic":"290556"},{"coNcm":"64039990","year":"2025","monthNumber":"01","country":"Estados Unidos","state":"Rio Grande do Sul","ncm":"Outros cal\u00e7ados sola exterior borracha\/pl\u00e1stico, de couro\/natural","headingCode":"6403","heading":"Cal\u00e7ado com sola exterior de borracha, pl\u00e1stico, couro natural ou reconstitu\u00eddo e parte superior de couro natural","metricFOB":"6101520","metricKG":"158929","metricStatistic":"306917"},{"coNcm":"64029990","year":"2025","monthNumber":"02","country":"Argentina","state":"Rio Grande do Sul","ncm":"Outr

In [3]:
#transforma o retorno em json
json_dados = response.json()
print(json_dados)

{'data': {'list': [{'coNcm': '64039990', 'year': '2025', 'monthNumber': '02', 'country': 'Estados Unidos', 'state': 'Rio Grande do Sul', 'ncm': 'Outros calçados sola exterior borracha/plástico, de couro/natural', 'headingCode': '6403', 'heading': 'Calçado com sola exterior de borracha, plástico, couro natural ou reconstituído e parte superior de couro natural', 'metricFOB': '6338562', 'metricKG': '133474', 'metricStatistic': '290556'}, {'coNcm': '64039990', 'year': '2025', 'monthNumber': '01', 'country': 'Estados Unidos', 'state': 'Rio Grande do Sul', 'ncm': 'Outros calçados sola exterior borracha/plástico, de couro/natural', 'headingCode': '6403', 'heading': 'Calçado com sola exterior de borracha, plástico, couro natural ou reconstituído e parte superior de couro natural', 'metricFOB': '6101520', 'metricKG': '158929', 'metricStatistic': '306917'}, {'coNcm': '64029990', 'year': '2025', 'monthNumber': '02', 'country': 'Argentina', 'state': 'Rio Grande do Sul', 'ncm': 'Outros calçados co

In [ ]:
#cria dataframe
normaliza_dados = pd.json_normalize(json_dados['data']['list'])
data_frame = pd.DataFrame(normaliza_dados)

In [15]:
#ajusta o tipo dos dados
data_frame = data_frame.astype({
    'year': int,
    'monthNumber': int,
    'coNcm': int,
    'metricFOB': float,
    'metricKG': float,
    'metricStatistic': float,
    'headingCode': int
})

In [16]:
#ordena os dados
data_frame = data_frame.sort_values(by='monthNumber', ascending=True)

In [17]:
#cria coluna dia
data_frame['dia'] = 1

In [18]:
#cria coluna data
data_frame['data'] = (data_frame['dia'].astype(str).str.zfill(2)+'/'+data_frame['monthNumber'].astype(str).str.zfill(2)+'/'+data_frame['year'].astype(str))

In [19]:
#ordena colunas
data_frame = data_frame[['year','monthNumber','dia','data','headingCode','heading','coNcm','ncm','country','state','metricFOB','metricKG','metricStatistic']]

In [20]:
#salva dados em excel
data_frame.to_excel('./dados_exportacao.xlsx', header=True, index=False,sheet_name='exportacao')